# Preparing Movie Plot Text

This notebook prepares movie plot summaries for NLP experiments.
We will inspect the source data before matching it to MovieLens.

Source: CMU Movie Summary Corpus, collected by David Bamman,
Brendan O’Connor, and Noah A. Smith.

Dataset: https://www.cs.cmu.edu/~ark/personas/
License: https://creativecommons.org/licenses/by-sa/3.0/us/

In [1]:
from pathlib import Path
import csv
import pandas as pd

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

cmu_dir = project_root / "data" / "raw" / "MovieSummaries"

In [2]:
metadata_columns = [
    "wikipedia_movie_id",
    "freebase_movie_id",
    "cmu_title",
    "release_date",
    "box_office_revenue",
    "runtime",
    "languages",
    "countries",
    "cmu_genres"
]

cmu_metadata = pd.read_csv(
    cmu_dir / "movie.metadata.tsv",
    sep="\t",
    header=None,
    names=metadata_columns,
    quoting=csv.QUOTE_NONE,
    encoding="utf-8"
)

print("Metadata shape:", cmu_metadata.shape)

cmu_metadata[
    ["wikipedia_movie_id", "cmu_title", "release_date"]
].head()

Metadata shape: (81741, 9)


,wikipedia_movie_id,cmu_title,release_date
0,975900,Ghosts of Mars,2001-08-24
1,3196793,Getting Away with Murder: The JonBenét Ramsey ...,2000-02-16
2,28463795,Brun bitter,1988
3,9363483,White Of The Eye,1987
4,261236,A Woman in Flames,1983


In [3]:
cmu_plots = pd.read_csv(
    cmu_dir / "plot_summaries.txt",
    sep="\t",
    header=None,
    names=["wikipedia_movie_id", "plot"],
    quoting=csv.QUOTE_NONE,
    encoding="utf-8"
)

print("Plot summaries shape:", cmu_plots.shape)

print("\nFirst plot — first 500 characters:")
print(cmu_plots["plot"].iloc[0][:500])

Plot summaries shape: (42306, 2)

First plot — first 500 characters:
Shlykov, a hard-working taxi driver and Lyosha, a saxophonist, develop a bizarre love-hate relationship, and despite their prejudices, realize they aren't so different after all.


In [4]:
for table_name, table in [
    ("Metadata", cmu_metadata),
    ("Plot summaries", cmu_plots)
]:
    print(table_name)
    print(
        "Missing movie IDs:",
        table["wikipedia_movie_id"].isna().sum()
    )
    print(
        "Extra rows with repeated movie IDs:",
        table["wikipedia_movie_id"].duplicated().sum()
    )
    print()

empty_plot_text = (
    cmu_plots["plot"].fillna("").str.strip().eq("")
)

plots_without_metadata = (
    ~cmu_plots["wikipedia_movie_id"].isin(
        cmu_metadata["wikipedia_movie_id"]
    )
)

print("Missing or blank plot text:", empty_plot_text.sum())
print(
    "Plot rows without matching metadata:",
    plots_without_metadata.sum()
)

Metadata
Missing movie IDs: 0
Extra rows with repeated movie IDs: 0

Plot summaries
Missing movie IDs: 0
Extra rows with repeated movie IDs: 0

Missing or blank plot text: 0
Plot rows without matching metadata: 99


The checks support a one-to-one join. The 99 summaries have movie IDs and usable text, but their IDs have no corresponding entry in the supplied metadata.

That’s approximately 0.23% of the summaries. We’ll keep them visible and flag their missing metadata.

We will merge the summaries and metadata

In [5]:
cmu_plot_catalog = cmu_plots.merge(
    cmu_metadata[
        ["wikipedia_movie_id", "cmu_title", "release_date"]
    ],
    on="wikipedia_movie_id",
    how="left",
    validate="one_to_one",
    indicator="metadata_match"
)

In [6]:
print("Plot rows before join:", len(cmu_plots))
print("Plot rows after join:", len(cmu_plot_catalog))

print(
    "Plots with matching metadata:",
    cmu_plot_catalog["metadata_match"].eq("both").sum()
)

print(
    "Plots without matching metadata:",
    cmu_plot_catalog["metadata_match"].eq("left_only").sum()
)

cmu_plot_catalog[
    ["wikipedia_movie_id", "cmu_title", "release_date", "metadata_match"]
].head()

Plot rows before join: 42306
Plot rows after join: 42306
Plots with matching metadata: 42207
Plots without matching metadata: 99


,wikipedia_movie_id,cmu_title,release_date,metadata_match
0,23890098,Taxi Blues,1990-09-07,both
1,31186339,The Hunger Games,2012-03-12,both
2,20663735,Narasimham,2000,both
3,2231378,The Lemon Drop Kid,1951-03-08,both
4,595909,A Cry in the Dark,1988-11-03,both


In [7]:
unmatched_plots = cmu_plot_catalog.loc[
    cmu_plot_catalog["metadata_match"] == "left_only",
    ["wikipedia_movie_id", "plot"]
].copy()

unmatched_plots.head()

,wikipedia_movie_id,plot
231,2862137,Sathyanarayan Murthy is a rough but kind man ...
540,33334420,{{more plot}} Mural tells the story of Zhu Xia...
1071,16758721,"The story is about a young man “Tony”, a 25 ye..."
1449,23217064,"Years before the events of the first film, Lo..."
2086,2746943,"A young 16-year-old girl, Zhu Ying Tai, manage..."


The join looks correct: 42,207 plots have matching metadata, and all 42,306 plot rows were preserved.

Next, we’ll prepare the release year and check whether titles and years are available. These will help us match CMU movies to MovieLens—using the year helps distinguish films with the same title.

Create a working table and extract the year

In [8]:
cmu_match_candidates = cmu_plot_catalog.loc[
    cmu_plot_catalog["metadata_match"] == "both"
].copy()

cmu_match_candidates["release_year"] = (
    cmu_match_candidates["release_date"]
    .astype("string")
    .str.strip()
    .str.extract(r"^(\d{4})(?:-|$)", expand=False)
    .astype("Int64")
)

In [9]:
has_title = (
    cmu_match_candidates["cmu_title"]
    .fillna("")
    .str.strip()
    .ne("")
)

has_release_year = (
    cmu_match_candidates["release_year"].notna()
)

print("Metadata-matched plots:", len(cmu_match_candidates))
print("Missing or blank titles:", (~has_title).sum())
print("Missing or unrecognized years:", (~has_release_year).sum())
print(
    "Rows with both title and year:",
    (has_title & has_release_year).sum()
)

cmu_match_candidates[
    ["wikipedia_movie_id", "cmu_title", "release_date", "release_year"]
].head()

Metadata-matched plots: 42207
Missing or blank titles: 0
Missing or unrecognized years: 2618
Rows with both title and year: 39589


,wikipedia_movie_id,cmu_title,release_date,release_year
0,23890098,Taxi Blues,1990-09-07,1990
1,31186339,The Hunger Games,2012-03-12,2012
2,20663735,Narasimham,2000,2000
3,2231378,The Lemon Drop Kid,1951-03-08,1951
4,595909,A Cry in the Dark,1988-11-03,1988


We now have 39,589 plots with a title and an extracted year for our first matching pass. The other records remain available in the earlier tables for later review.

Next, we’ll create comparable title keys in both datasets.

In [10]:
cmu_match_ready = cmu_match_candidates.loc[
    has_title & has_release_year
].copy()

movielens_movies = pd.read_csv(
    project_root / "data" / "raw" / "ml-latest-small" / "movies.csv"
)

movielens_movies["release_year"] = (
    movielens_movies["title"]
    .str.extract(r"\((\d{4})\)\s*$", expand=False)
    .astype("Int64")
)

movielens_movies["title_without_year"] = (
    movielens_movies["title"]
    .str.replace(r"\s*\(\d{4}\)\s*$", "", regex=True)
    .str.strip()
)

In [11]:
import re

def normalize_title(title):
    """Standardize a movie title for matching across datasets."""
    title = title.casefold().strip()
    title = re.sub(r"\s+", " ", title)
    title = re.sub(r"^(.+),\s*(the|a|an)$", r"\2 \1", title)
    return title


cmu_match_ready["title_key"] = (
    cmu_match_ready["cmu_title"].map(normalize_title)
)

movielens_movies["title_key"] = (
    movielens_movies["title_without_year"].map(normalize_title)
)

In [12]:
print("CMU movies prepared:", len(cmu_match_ready))
print("MovieLens movies:", len(movielens_movies))
print(
    "MovieLens movies without an extracted year:",
    movielens_movies["release_year"].isna().sum()
)

display(
    cmu_match_ready[
        ["cmu_title", "title_key", "release_year"]
    ].head()
)

display(
    movielens_movies[
        ["movieId", "title", "title_key", "release_year"]
    ].head()
)

CMU movies prepared: 39589
MovieLens movies: 9742
MovieLens movies without an extracted year: 13


,cmu_title,title_key,release_year
0,Taxi Blues,taxi blues,1990
1,The Hunger Games,the hunger games,2012
2,Narasimham,narasimham,2000
3,The Lemon Drop Kid,the lemon drop kid,1951
4,A Cry in the Dark,a cry in the dark,1988


,movieId,title,title_key,release_year
0,1,Toy Story (1995),toy story,1995
1,2,Jumanji (1995),jumanji,1995
2,3,Grumpier Old Men (1995),grumpier old men,1995
3,4,Waiting to Exhale (1995),waiting to exhale,1995
4,5,Father of the Bride Part II (1995),father of the bride part ii,1995


let’s check whether title_key plus release_year uniquely identifies a movie within each dataset.

Unique movie IDs don’t guarantee unique title-and-year combinations. Two different films could share both.

Prepare the MovieLens matching table

In [13]:
movielens_match_ready = movielens_movies.loc[
    movielens_movies["release_year"].notna()
].copy()

print("MovieLens rows ready for matching:", len(movielens_match_ready))

MovieLens rows ready for matching: 9729


In [14]:
match_columns = ["title_key", "release_year"]

cmu_repeated_keys = cmu_match_ready.duplicated(
    subset=match_columns,
    keep=False
)

movielens_repeated_keys = movielens_match_ready.duplicated(
    subset=match_columns,
    keep=False
)

for name, table, repeated_keys in [
    ("CMU", cmu_match_ready, cmu_repeated_keys),
    ("MovieLens", movielens_match_ready, movielens_repeated_keys)
]:
    print(name)
    print(
        "Blank title keys:",
        table["title_key"].fillna("").str.strip().eq("").sum()
    )
    print(
        "Rows belonging to repeated title/year combinations:",
        repeated_keys.sum()
    )
    print()

CMU
Blank title keys: 0
Rows belonging to repeated title/year combinations: 83

MovieLens
Blank title keys: 0
Rows belonging to repeated title/year combinations: 10



In [15]:
display(
    cmu_match_ready.loc[
        cmu_repeated_keys,
        ["wikipedia_movie_id", "cmu_title", "title_key", "release_year"]
    ].sort_values(match_columns).head(10)
)

display(
    movielens_match_ready.loc[
        movielens_repeated_keys,
        ["movieId", "title", "title_key", "release_year"]
    ].sort_values(match_columns).head(10)
)

,wikipedia_movie_id,cmu_title,title_key,release_year
288,33431914,180,180,2011
28230,28375513,180,180,2011
5806,31964514,A Better Life,a better life,2011
37966,33820074,A Better Life,a better life,2011
7887,14969658,Angel,angel,2007
29833,12864134,Angel,angel,2007
6333,25489035,Bad Karma,bad karma,2002
40507,25080803,Bad Karma,bad karma,2002
3804,31249953,Black Gold,black gold,2011
21514,33245032,Black Gold,black gold,2011


,movieId,title,title_key,release_year
4169,6003,Confessions of a Dangerous Mind (2002),confessions of a dangerous mind,2002
9106,144606,Confessions of a Dangerous Mind (2002),confessions of a dangerous mind,2002
650,838,Emma (1996),emma,1996
5601,26958,Emma (1996),emma,1996
5854,32600,Eros (2004),eros,2004
9135,147002,Eros (2004),eros,2004
2141,2851,Saturn 3 (1980),saturn 3,1980
9468,168358,Saturn 3 (1980),saturn 3,1980
5931,34048,War of the Worlds (2005),war of the worlds,2005
6932,64997,War of the Worlds (2005),war of the worlds,2005


In [16]:
display(
    cmu_match_ready.loc[
        cmu_repeated_keys,
        ["wikipedia_movie_id", "cmu_title", "title_key", "release_year"]
    ].sort_values(match_columns).head(10)
)

display(
    movielens_match_ready.loc[
        movielens_repeated_keys,
        ["movieId", "title", "title_key", "release_year"]
    ].sort_values(match_columns).head(10)
)

,wikipedia_movie_id,cmu_title,title_key,release_year
288,33431914,180,180,2011
28230,28375513,180,180,2011
5806,31964514,A Better Life,a better life,2011
37966,33820074,A Better Life,a better life,2011
7887,14969658,Angel,angel,2007
29833,12864134,Angel,angel,2007
6333,25489035,Bad Karma,bad karma,2002
40507,25080803,Bad Karma,bad karma,2002
3804,31249953,Black Gold,black gold,2011
21514,33245032,Black Gold,black gold,2011


,movieId,title,title_key,release_year
4169,6003,Confessions of a Dangerous Mind (2002),confessions of a dangerous mind,2002
9106,144606,Confessions of a Dangerous Mind (2002),confessions of a dangerous mind,2002
650,838,Emma (1996),emma,1996
5601,26958,Emma (1996),emma,1996
5854,32600,Eros (2004),eros,2004
9135,147002,Eros (2004),eros,2004
2141,2851,Saturn 3 (1980),saturn 3,1980
9468,168358,Saturn 3 (1980),saturn 3,1980
5931,34048,War of the Worlds (2005),war of the worlds,2005
6932,64997,War of the Worlds (2005),war of the worlds,2005


In [17]:
cmu_unique_keys = cmu_match_ready.loc[
    ~cmu_repeated_keys
].copy()

movielens_unique_keys = movielens_match_ready.loc[
    ~movielens_repeated_keys
].copy()

print("CMU rows excluded as ambiguous:", cmu_repeated_keys.sum())
print("MovieLens rows excluded as ambiguous:", movielens_repeated_keys.sum())

print("CMU rows with unique keys:", len(cmu_unique_keys))
print("MovieLens rows with unique keys:", len(movielens_unique_keys))

CMU rows excluded as ambiguous: 83
MovieLens rows excluded as ambiguous: 10
CMU rows with unique keys: 39506
MovieLens rows with unique keys: 9719


In [18]:
initial_plot_matches = movielens_unique_keys.merge(
    cmu_unique_keys[
        [
            "title_key",
            "release_year",
            "wikipedia_movie_id",
            "cmu_title",
            "plot"
        ]
    ],
    on=match_columns,
    how="left",
    validate="one_to_one",
    indicator="plot_match"
)

In [19]:
matched = initial_plot_matches["plot_match"].eq("both")

print("MovieLens rows before join:", len(movielens_unique_keys))
print("Rows after join:", len(initial_plot_matches))
print("Movies matched to plots:", matched.sum())
print("Eligible movies without a match:", (~matched).sum())

print(
    "Initial coverage of all MovieLens movies:",
    f"{matched.sum() / len(movielens_movies):.1%}"
)

display(
    initial_plot_matches.loc[
        matched,
        ["movieId", "title", "cmu_title", "release_year"]
    ].head(10)
)

MovieLens rows before join: 9719
Rows after join: 9719
Movies matched to plots: 5857
Eligible movies without a match: 3862
Initial coverage of all MovieLens movies: 60.1%


,movieId,title,cmu_title,release_year
0,1,Toy Story (1995),Toy Story,1995
1,2,Jumanji (1995),Jumanji,1995
2,3,Grumpier Old Men (1995),Grumpier Old Men,1995
3,4,Waiting to Exhale (1995),Waiting to Exhale,1995
4,5,Father of the Bride Part II (1995),Father of the Bride Part II,1995
5,6,Heat (1995),Heat,1995
6,7,Sabrina (1995),Sabrina,1995
7,8,Tom and Huck (1995),Tom and Huck,1995
8,9,Sudden Death (1995),Sudden Death,1995
9,10,GoldenEye (1995),GoldenEye,1995


In [20]:
unmatched_movielens = initial_plot_matches.loc[
    initial_plot_matches["plot_match"].eq("left_only"),
    ["movieId", "title", "title_without_year", "title_key", "release_year"]
].copy()

display(unmatched_movielens.head(20))

,movieId,title,title_without_year,title_key,release_year
25,26,Othello (1995),Othello,othello,1995
28,29,"City of Lost Children, The (Cité des enfants p...","City of Lost Children, The (Cité des enfants p...","city of lost children, the (cité des enfants p...",1995
29,30,Shanghai Triad (Yao a yao yao dao waipo qiao) ...,Shanghai Triad (Yao a yao yao dao waipo qiao),shanghai triad (yao a yao yao dao waipo qiao),1995
31,32,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),Twelve Monkeys (a.k.a. 12 Monkeys),twelve monkeys (a.k.a. 12 monkeys),1995
36,40,"Cry, the Beloved Country (1995)","Cry, the Beloved Country","cry, the beloved country",1995
37,41,Richard III (1995),Richard III,richard iii,1995
42,46,How to Make an American Quilt (1995),How to Make an American Quilt,how to make an american quilt,1995
43,47,Seven (a.k.a. Se7en) (1995),Seven (a.k.a. Se7en),seven (a.k.a. se7en),1995
45,49,When Night Is Falling (1995),When Night Is Falling,when night is falling,1995
46,50,"Usual Suspects, The (1995)","Usual Suspects, The",the usual suspects,1995


In [21]:
display(
    unmatched_movielens.sample(
        n=min(15, len(unmatched_movielens)),
        random_state=42
    ).sort_values("release_year")
)

,movieId,title,title_without_year,title_key,release_year
4347,6358,Kiss Me Kate (1953),Kiss Me Kate,kiss me kate,1953
5462,26159,Tokyo Drifter (Tôkyô nagaremono) (1966),Tokyo Drifter (Tôkyô nagaremono),tokyo drifter (tôkyô nagaremono),1966
110,128,Jupiter's Wife (1994),Jupiter's Wife,jupiter's wife,1994
4502,6667,Waco: The Rules of Engagement (1997),Waco: The Rules of Engagement,waco: the rules of engagement,1997
1311,1760,Spice World (1997),Spice World,spice world,1997
5236,8587,Mayor of the Sunset Strip (2003),Mayor of the Sunset Strip,mayor of the sunset strip,2003
4916,7375,"Prince & Me, The (2004)","Prince & Me, The",the prince & me,2004
6947,65651,Fire and Ice (2008),Fire and Ice,fire and ice,2008
7444,81535,Saw VII 3D - The Final Chapter (2010),Saw VII 3D - The Final Chapter,saw vii 3d - the final chapter,2010
8275,105801,Escape From Tomorrow (2013),Escape From Tomorrow,escape from tomorrow,2013


In [22]:
def make_main_title_key(title):
    """Remove parenthetical text, then normalize the remaining title."""
    main_title = re.sub(r"\([^()]*\)", "", title)
    return normalize_title(main_title)


movielens_match_ready["main_title_key"] = (
    movielens_match_ready["title_without_year"].map(make_main_title_key)
)

cmu_match_ready["main_title_key"] = (
    cmu_match_ready["cmu_title"].map(make_main_title_key)
)

In [23]:
changed_unmatched = movielens_match_ready.loc[
    movielens_match_ready["movieId"].isin(
        unmatched_movielens["movieId"]
    )
    & movielens_match_ready["main_title_key"].ne(
        movielens_match_ready["title_key"]
    ),
    ["movieId", "title", "title_key", "main_title_key", "release_year"]
]

print("Unmatched movies with a changed key:", len(changed_unmatched))

display(changed_unmatched.head(15))

Unmatched movies with a changed key: 1072


,movieId,title,title_key,main_title_key,release_year
28,29,"City of Lost Children, The (Cité des enfants p...","city of lost children, the (cité des enfants p...",the city of lost children,1995
29,30,Shanghai Triad (Yao a yao yao dao waipo qiao) ...,shanghai triad (yao a yao yao dao waipo qiao),shanghai triad,1995
31,32,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),twelve monkeys (a.k.a. 12 monkeys),twelve monkeys,1995
43,47,Seven (a.k.a. Se7en) (1995),seven (a.k.a. se7en),seven,1995
52,58,"Postman, The (Postino, Il) (1994)","postman, the (postino, il)",the postman,1994
60,68,French Twist (Gazon maudit) (1995),french twist (gazon maudit),french twist,1995
72,80,"White Balloon, The (Badkonake sefid) (1995)","white balloon, the (badkonake sefid)",the white balloon,1995
74,82,Antonia's Line (Antonia) (1995),antonia's line (antonia),antonia's line,1995
86,97,"Hate (Haine, La) (1995)","hate (haine, la)",hate,1995
94,106,Nobody Loves Me (Keiner liebt mich) (1994),nobody loves me (keiner liebt mich),nobody loves me,1994


In [24]:
main_match_columns = ["main_title_key", "release_year"]

cmu_main_repeated = cmu_match_ready.duplicated(
    subset=main_match_columns,
    keep=False
)

movielens_main_repeated = movielens_match_ready.duplicated(
    subset=main_match_columns,
    keep=False
)

for name, table, repeated in [
    ("CMU", cmu_match_ready, cmu_main_repeated),
    ("MovieLens", movielens_match_ready, movielens_main_repeated)
]:
    print(name)
    print(
        "Blank main-title keys:",
        table["main_title_key"].eq("").sum()
    )
    print(
        "Rows in repeated main-title/year combinations:",
        repeated.sum()
    )
    print()

CMU
Blank main-title keys: 1
Rows in repeated main-title/year combinations: 83

MovieLens
Blank main-title keys: 0
Rows in repeated main-title/year combinations: 10



In [25]:
exact_plot_matches = initial_plot_matches.loc[
    initial_plot_matches["plot_match"].eq("both")
].copy()

movielens_second_pass = movielens_match_ready.loc[
    ~movielens_main_repeated
    & movielens_match_ready["main_title_key"].ne("")
    & movielens_match_ready["movieId"].isin(
        unmatched_movielens["movieId"]
    )
].copy()

cmu_second_pass = cmu_match_ready.loc[
    ~cmu_main_repeated
    & cmu_match_ready["main_title_key"].ne("")
    & ~cmu_match_ready["wikipedia_movie_id"].isin(
        exact_plot_matches["wikipedia_movie_id"]
    )
].copy()

In [26]:
main_title_proposals = movielens_second_pass.merge(
    cmu_second_pass[
        [
            "main_title_key",
            "release_year",
            "wikipedia_movie_id",
            "cmu_title",
            "plot"
        ]
    ],
    on=main_match_columns,
    how="inner",
    validate="one_to_one"
)

main_title_proposals["match_method"] = "main_title_year"

In [27]:
print("Original matches:", len(exact_plot_matches))
print("Additional proposed matches:", len(main_title_proposals))

potential_total = len(exact_plot_matches) + len(main_title_proposals)

print("Potential total matches:", potential_total)
print(
    "Potential coverage of all MovieLens movies:",
    f"{potential_total / len(movielens_movies):.1%}"
)

with pd.option_context("display.max_colwidth", None):
    display(
        main_title_proposals[
            [
                "movieId",
                "title",
                "cmu_title",
                "main_title_key",
                "release_year"
            ]
        ].head(20)
    )

Original matches: 5857
Additional proposed matches: 585
Potential total matches: 6442
Potential coverage of all MovieLens movies: 66.1%


,movieId,title,cmu_title,main_title_key,release_year
0,29,"City of Lost Children, The (Cité des enfants perdus, La) (1995)",The City of Lost Children,the city of lost children,1995
1,30,Shanghai Triad (Yao a yao yao dao waipo qiao) (1995),Shanghai Triad,shanghai triad,1995
2,32,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),Twelve Monkeys,twelve monkeys,1995
3,47,Seven (a.k.a. Se7en) (1995),Seven,seven,1995
4,68,French Twist (Gazon maudit) (1995),French Twist,french twist,1995
5,80,"White Balloon, The (Badkonake sefid) (1995)",The White Balloon,the white balloon,1995
6,112,Rumble in the Bronx (Hont faan kui) (1995),Rumble in the Bronx,rumble in the bronx,1995
7,123,Chungking Express (Chung Hing sam lam) (1994),Chungking Express,chungking express,1994
8,154,Beauty of the Day (Belle de jour) (1967),Beauty of the Day,beauty of the day,1967
9,199,"Umbrellas of Cherbourg, The (Parapluies de Cherbourg, Les) (1964)",The Umbrellas of Cherbourg,the umbrellas of cherbourg,1964


In [28]:
exact_plot_matches["match_method"] = "normalized_title_year"

match_output_columns = [
    "movieId",
    "title",
    "release_year",
    "wikipedia_movie_id",
    "cmu_title",
    "plot",
    "match_method"
]

movie_plot_matches = pd.concat(
    [
        exact_plot_matches[match_output_columns],
        main_title_proposals[match_output_columns]
    ],
    ignore_index=True
)

movie_plot_matches["wikipedia_movie_id"] = (
    movie_plot_matches["wikipedia_movie_id"].astype("Int64")
)

In [29]:
assert len(movie_plot_matches) == (
    len(exact_plot_matches) + len(main_title_proposals)
)

assert movie_plot_matches["movieId"].notna().all()
assert movie_plot_matches["wikipedia_movie_id"].notna().all()

assert movie_plot_matches["movieId"].is_unique
assert movie_plot_matches["wikipedia_movie_id"].is_unique

assert (
    movie_plot_matches["plot"]
    .fillna("")
    .str.strip()
    .ne("")
    .all()
)

print("Matched movies:", len(movie_plot_matches))
print(
    "Coverage:",
    f"{len(movie_plot_matches) / len(movielens_movies):.1%}"
)

print("\nMatches by method:")
print(movie_plot_matches["match_method"].value_counts())

Matched movies: 6442
Coverage: 66.1%

Matches by method:
match_method
normalized_title_year    5857
main_title_year           585
Name: count, dtype: int64


In [30]:
plot_sample = movie_plot_matches.loc[
    movie_plot_matches["match_method"].eq("main_title_year")
].sample(n=3, random_state=42)

for row in plot_sample.itertuples(index=False):
    print(f"MovieLens: {row.title}")
    print(f"CMU: {row.cmu_title}")
    print(f"Plot excerpt: {row.plot[:600]}")
    print()

MovieLens: Accidental Spy, The (Dak miu mai shing) (2001)
CMU: The Accidental Spy
Plot excerpt: A news reporter covers a story in Turkey where many people have mysteriously died, seemingly from pneumonia. In Hong Kong, Buck Yuen works as an exercise equipment salesman. After an unsuccessful day at work, he heads out through the shopping mall and intuitively knows that a robbery is about to take place. Buck thwarts the robbers plans, relieving them of the money they have stolen and returning it to the police. Later, a stranger approaches Buck, eager to speak to the hero who foiled the robbery. The stranger goes by the name of Manny Liu and he tells Buck that he is rounding up several men 

MovieLens: Common Thread, A (a.k.a. Sequins) (Brodeuses) (2004)
CMU: A Common Thread
Plot excerpt: When Claire  learns that she is five months pregnant at the age of seventeen, she decides to keep the baby and not to tell the father . Instead, she quits her current job at a local supermarket and goes 

The checks passed, and the excerpts look like usable narrative text. They show some extra spaces, but nothing in this sample that requires substantial cleaning before our first text model.

Next, we need plot coverage among the movies our recommender can actually recommend. The 66.1% figure includes the entire MovieLens catalog; our earlier models used only movies present in training.

Load the same training split into notebook 4

Copy and run the cells from notebook 3 that load or create train_ratings, including any required setup. Reuse the existing split logic exactly.

In [31]:
from pathlib import Path

import pandas as pd

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

data_dir = project_root / "data" / "raw" / "ml-latest-small"

movies = pd.read_csv(data_dir / "movies.csv")
ratings = pd.read_csv(data_dir / "ratings.csv")

ratings["rated_at"] = pd.to_datetime(
    ratings["timestamp"],
    unit="s",
    utc=True
)

dates = ratings["rated_at"]

train_cutoff = dates.quantile(0.60)
validation_cutoff = dates.quantile(0.80)

train_ratings = ratings.loc[
    dates <= train_cutoff
].copy()

validation_ratings = ratings.loc[
    (dates > train_cutoff) & (dates <= validation_cutoff)
].copy()

test_ratings = ratings.loc[
    dates > validation_cutoff
].copy()

print("Training rows:", len(train_ratings))
print("Validation rows:", len(validation_ratings))
print("Test rows:", len(test_ratings))

Training rows: 60502
Validation rows: 20167
Test rows: 20167


In [32]:
print("Training ratings:", len(train_ratings))
print("Training movies:", train_ratings["movieId"].nunique())

Training ratings: 60502
Training movies: 6233


In [33]:
training_movie_catalog = movielens_movies.loc[
    movielens_movies["movieId"].isin(train_ratings["movieId"]),
    ["movieId", "title", "genres"]
].copy()

training_plot_catalog = training_movie_catalog.merge(
    movie_plot_matches[
        [
            "movieId",
            "wikipedia_movie_id",
            "cmu_title",
            "plot",
            "match_method"
        ]
    ],
    on="movieId",
    how="left",
    validate="one_to_one"
)

training_plot_catalog["has_plot"] = (
    training_plot_catalog["plot"]
    .fillna("")
    .str.strip()
    .ne("")
)

assert len(training_plot_catalog) == train_ratings["movieId"].nunique()

In [34]:
print("Training candidate movies:", len(training_plot_catalog))
print("With plots:", training_plot_catalog["has_plot"].sum())
print("Without plots:", (~training_plot_catalog["has_plot"]).sum())

print(
    "Training candidate plot coverage:",
    f"{training_plot_catalog['has_plot'].mean():.1%}"
)

display(
    training_plot_catalog.loc[
        ~training_plot_catalog["has_plot"],
        ["movieId", "title"]
    ].head(10)
)

Training candidate movies: 6233
With plots: 5036
Without plots: 1197
Training candidate plot coverage: 80.8%


,movieId,title
25,26,Othello (1995)
36,40,"Cry, the Beloved Country (1995)"
37,41,Richard III (1995)
42,46,How to Make an American Quilt (1995)
45,49,When Night Is Falling (1995)
46,50,"Usual Suspects, The (1995)"
50,55,Georgia (1995)
52,58,"Postman, The (Postino, Il) (1994)"
64,72,Kicking and Screaming (1995)
65,73,"Misérables, Les (1995)"


In [35]:
text_movies = (
    training_plot_catalog.loc[
        training_plot_catalog["has_plot"]
    ]
    .sort_values("movieId")
    .reset_index(drop=True)
    .copy()
)

print("Movies available for text features:", len(text_movies))

Movies available for text features: 5036


In [36]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    min_df=2,
    norm="l2"
)

plot_tfidf = tfidf_vectorizer.fit_transform(text_movies["plot"])

print("TF-IDF matrix shape:", plot_tfidf.shape)
print(
    "Movies with no retained words:",
    (plot_tfidf.getnnz(axis=1) == 0).sum()
)

TF-IDF matrix shape: (5036, 31609)
Movies with no retained words: 0


In [37]:
toy_story_row = text_movies.index[
    text_movies["movieId"].eq(1)
][0]

toy_story_weights = pd.Series(
    plot_tfidf[toy_story_row].toarray().ravel(),
    index=tfidf_vectorizer.get_feature_names_out(),
    name="tfidf_weight"
)

toy_story_weights.nlargest(15)

buzz         0.614665
woody        0.572134
andy         0.315047
toys         0.246549
sid          0.143422
rc           0.125132
planet       0.089399
toy          0.080528
pizza        0.075495
claw         0.061467
lightyear    0.053283
scud         0.051464
rocket       0.050896
moving       0.050378
potato       0.047081
Name: tfidf_weight, dtype: float64

We now have 5,036 movies represented by 31,609 word features, and every movie has at least one nonzero feature.

Toy Story’s vector is dominated by character names—Buzz, Woody, and Andy—followed by words such as “toys.” These weights reflect both repetition within its plot and how widespread each word is across the other plots.

This gives us a useful hypothesis: shared character names may help identify sequels, but could also connect unrelated movies with similarly named characters. Let’s inspect the actual results.

In [38]:
from sklearn.metrics.pairwise import cosine_similarity

toy_story_similarities = cosine_similarity(
    plot_tfidf[toy_story_row],
    plot_tfidf
).ravel()

print("Similarity scores calculated:", len(toy_story_similarities))
print(
    "Toy Story similarity to itself:",
    toy_story_similarities[toy_story_row]
)

Similarity scores calculated: 5036
Toy Story similarity to itself: 1.0000000000000002


In [39]:
similar_movies = text_movies[["movieId", "title"]].copy()

similar_movies["similarity"] = toy_story_similarities

training_rating_counts = train_ratings["movieId"].value_counts()

similar_movies["training_rating_count"] = (
    similar_movies["movieId"].map(training_rating_counts)
)

similar_movies = (
    similar_movies.loc[similar_movies["movieId"].ne(1)]
    .sort_values(
        ["similarity", "training_rating_count", "movieId"],
        ascending=[False, False, True]
    )
    .head(10)
    .reset_index(drop=True)
)

similar_movies

,movieId,title,similarity,training_rating_count
0,3114,Toy Story 2 (1999),0.771482,64
1,2041,Condorman (1981),0.343202,3
2,3628,Flying Tigers (1942),0.280637,2
3,51575,Wild Hogs (2007),0.264628,2
4,1863,Major League: Back to the Minors (1998),0.262346,6
5,55805,Before the Devil Knows You're Dead (2007),0.250632,1
6,35836,"40-Year-Old Virgin, The (2005)",0.248084,28
7,318,"Shawshank Redemption, The (1994)",0.246460,177
8,2793,"American Werewolf in Paris, An (1997)",0.236362,9
9,2884,Dog Park (1998),0.220531,1


This compares Toy Story with:
- Toy Story 2: Our strongest match.
- Condorman: The second-ranked result.
- The Shawshank Redemption: A seemingly surprising match.
Why .multiply()? It multiplies corresponding word weights individually. That lets us inspect the ingredients of the similarity score instead of seeing only the final total.

The printed total uses all word contributions; the table displays only the eight largest.

If names such as “Woody” or “Andy” dominate the unexpected matches, we’ll have direct evidence that shared names are driving them. Share these three outputs, and we’ll interpret what that means for our first text recommender.

In [40]:
feature_names = tfidf_vectorizer.get_feature_names_out()

for comparison_movie_id in [3114, 2041, 318]:
    comparison_row = text_movies.index[
        text_movies["movieId"].eq(comparison_movie_id)
    ][0]

    word_products = plot_tfidf[toy_story_row].multiply(
        plot_tfidf[comparison_row]
    )

    contributions = pd.Series(
        word_products.toarray().ravel(),
        index=feature_names,
        name="similarity_contribution"
    )

    print(text_movies.loc[comparison_row, "title"])
    print(f"Total similarity: {contributions.sum():.6f}")

    display(
        contributions.loc[contributions > 0].nlargest(8)
    )

Toy Story 2 (1999)
Total similarity: 0.771482


woody        0.350987
buzz         0.248692
toys         0.076041
andy         0.057647
toy          0.016572
lightyear    0.003298
arm          0.002721
pizza        0.001655
Name: similarity_contribution, dtype: float64

Condorman (1981)
Total similarity: 0.343202


woody      0.338556
mission    0.000755
drives     0.000423
rescue     0.000404
return     0.000383
fly        0.000310
test       0.000291
named      0.000278
Name: similarity_contribution, dtype: float64

Shawshank Redemption, The (1994)
Total similarity: 0.246460


andy          0.242373
containing    0.000544
vicious       0.000388
murdering     0.000368
life          0.000347
receives      0.000336
resulting     0.000233
finds         0.000215
Name: similarity_contribution, dtype: float64

This confirms that shared character names drive the unexpected matches:
Comparison	Dominant shared word	Contribution	Share of total similarity
Toy Story → Condorman	woody	0.338556	98.6%
Toy Story → Shawshank	andy	0.242373	98.3%


TF-IDF treats each occurrence of “Andy” as the same feature; it cannot distinguish different characters with that name. For Toy Story 2, shared names help identify a related movie. Removing names could therefore improve some matches while weakening others.

We’ll keep this version as our first text baseline and evaluate it before changing its features.

Next, let’s generate personalized recommendations for user 1, using the same average-similarity approach

Find the user’s liked movies with available plots

In [41]:
example_user_id = 1

user_history = train_ratings.loc[
    train_ratings["userId"].eq(example_user_id)
]

liked_movie_ids = user_history.loc[
    user_history["rating"] >= 4.0,
    "movieId"
]

liked_text_rows = text_movies.index[
    text_movies["movieId"].isin(liked_movie_ids)
].to_numpy()

print("Liked training movies:", len(liked_movie_ids))
print("Liked movies with plot features:", len(liked_text_rows))
print(
    "Liked movies without plot features:",
    len(liked_movie_ids) - len(liked_text_rows)
)

Liked training movies: 200
Liked movies with plot features: 175
Liked movies without plot features: 25


Score and rank the full candidate catalog

In [42]:
text_recommendations = training_plot_catalog.loc[
    ~training_plot_catalog["movieId"].isin(user_history["movieId"]),
    ["movieId", "title", "has_plot"]
].copy()

text_recommendations["score"] = 0.0

if len(liked_text_rows) > 0:
    similarities = cosine_similarity(
        plot_tfidf[liked_text_rows],
        plot_tfidf
    )

    text_scores = pd.Series(
        similarities.mean(axis=0),
        index=text_movies["movieId"]
    )

    text_recommendations["score"] = (
        text_recommendations["movieId"]
        .map(text_scores)
        .fillna(0.0)
    )

training_rating_counts = train_ratings["movieId"].value_counts()

text_recommendations["training_rating_count"] = (
    text_recommendations["movieId"].map(training_rating_counts)
)

text_recommendations = (
    text_recommendations.sort_values(
        ["score", "training_rating_count", "movieId"],
        ascending=[False, False, True]
    )
    .head(10)
    .reset_index(drop=True)
)

text_recommendations

,movieId,title,has_plot,score,training_rating_count
0,2376,"View to a Kill, A (1985)",True,0.029131,16
1,5108,John Q (2002),True,0.028747,7
2,2989,For Your Eyes Only (1981),True,0.027795,15
3,10,GoldenEye (1995),True,0.027629,113
4,1911,Dr. Dolittle (1998),True,0.025480,22
5,7569,You Only Live Twice (1967),True,0.025395,6
6,27660,"Animatrix, The (2003)",True,0.025307,10
7,3635,"Spy Who Loved Me, The (1977)",True,0.025223,14
8,20,Money Train (1995),True,0.025153,11
9,4005,"Living Daylights, The (1987)",True,0.024418,8


User 1’s scores now average 175 plot comparisons per candidate; the 25 liked movies without plots don’t contribute. Several Bond titles appear, but we’d need to inspect the contributing comparisons to determine what drives them.

The scores are much smaller than the genre scores because the feature representations differ. We’ll compare the models using validation Recall@10, rather than their raw similarity scores.

Next, let’s package this calculation into a reusable function.

In [43]:
def recommend_text_based(
    user_id,
    training_data,
    text_movies,
    plot_tfidf,
    candidate_catalog,
    n=10
):
    """Recommend by mean plot similarity, with popularity as a fallback."""

    user_history = training_data.loc[
        training_data["userId"].eq(user_id)
    ]

    liked_movie_ids = user_history.loc[
        user_history["rating"] >= 4.0,
        "movieId"
    ]

    liked_text_rows = text_movies.index[
        text_movies["movieId"].isin(liked_movie_ids)
    ].to_numpy()

    recommendations = candidate_catalog.loc[
        ~candidate_catalog["movieId"].isin(user_history["movieId"]),
        ["movieId", "title", "has_plot"]
    ].copy()

    recommendations["score"] = 0.0

    if len(liked_text_rows) > 0:
        similarities = cosine_similarity(
            plot_tfidf[liked_text_rows],
            plot_tfidf
        )

        scores = pd.Series(
            similarities.mean(axis=0),
            index=text_movies["movieId"]
        )

        recommendations["score"] = (
            recommendations["movieId"].map(scores).fillna(0.0)
        )

    rating_counts = training_data["movieId"].value_counts()

    recommendations["training_rating_count"] = (
        recommendations["movieId"].map(rating_counts)
    )

    return recommendations.sort_values(
        ["score", "training_rating_count", "movieId"],
        ascending=[False, False, True]
    ).head(n).reset_index(drop=True)

Expected behavior:
- User 1: The same ten movies and scores as your manual calculation.
- User 2: Popularity recommendations with scores of zero, because this user has no training history.

In [44]:
for example_user_id in [1, 2]:
    print("Recommendations for user:", example_user_id)

    display(
        recommend_text_based(
            user_id=example_user_id,
            training_data=train_ratings,
            text_movies=text_movies,
            plot_tfidf=plot_tfidf,
            candidate_catalog=training_plot_catalog,
            n=10
        )
    )

Recommendations for user: 1


,movieId,title,has_plot,score,training_rating_count
0,2376,"View to a Kill, A (1985)",True,0.029131,16
1,5108,John Q (2002),True,0.028747,7
2,2989,For Your Eyes Only (1981),True,0.027795,15
3,10,GoldenEye (1995),True,0.027629,113
4,1911,Dr. Dolittle (1998),True,0.025480,22
5,7569,You Only Live Twice (1967),True,0.025395,6
6,27660,"Animatrix, The (2003)",True,0.025307,10
7,3635,"Spy Who Loved Me, The (1977)",True,0.025223,14
8,20,Money Train (1995),True,0.025153,11
9,4005,"Living Daylights, The (1987)",True,0.024418,8


Recommendations for user: 2


,movieId,title,has_plot,score,training_rating_count
0,296,Pulp Fiction (1994),True,0.0,201
1,356,Forrest Gump (1994),True,0.0,193
2,480,Jurassic Park (1993),True,0.0,185
3,318,"Shawshank Redemption, The (1994)",True,0.0,177
4,593,"Silence of the Lambs, The (1991)",True,0.0,177
5,110,Braveheart (1995),True,0.0,174
6,589,Terminator 2: Judgment Day (1991),True,0.0,166
7,457,"Fugitive, The (1993)",True,0.0,160
8,592,Batman (1989),True,0.0,160
9,780,Independence Day (a.k.a. ID4) (1996),True,0.0,157


Now we’ll measure validation Recall@10, using the same evaluation rules as the earlier models.

In [45]:
validation_positives = validation_ratings.loc[
    validation_ratings["rating"] >= 4.0
].copy()

text_based_rows = []

for user_id, user_positives in validation_positives.groupby("userId"):
    positive_movie_ids = user_positives["movieId"]

    recommendations = recommend_text_based(
        user_id=user_id,
        training_data=train_ratings,
        text_movies=text_movies,
        plot_tfidf=plot_tfidf,
        candidate_catalog=training_plot_catalog,
        n=10
    )

    hits = recommendations["movieId"].isin(
        positive_movie_ids
    ).sum()

    text_based_rows.append({
        "userId": user_id,
        "positive_movie_count": len(positive_movie_ids),
        "hits_at_10": hits,
        "recall_at_10": hits / len(positive_movie_ids)
    })

text_based_results = pd.DataFrame(text_based_rows)

print("Users evaluated:", len(text_based_results))
print(
    "Mean Recall@10:",
    text_based_results["recall_at_10"].mean()
)

Users evaluated: 165
Mean Recall@10: 0.037607967210118935


In [46]:
text_based_results["has_training_history"] = (
    text_based_results["userId"].isin(train_ratings["userId"])
)

text_group_summary = (
    text_based_results.groupby("has_training_history")
    .agg(
        users=("userId", "size"),
        mean_recall_at_10=("recall_at_10", "mean")
    )
)

text_group_summary

,users,mean_recall_at_10
has_training_history,,
False,140,0.041847
True,25,0.013870


## Plot TF-IDF recommender: validation findings

Plot summaries were matched to 6,442 MovieLens movies using unique
normalized-title/year and main-title/year matching rules.

Within the 6,233 training candidates, 5,036 movies had matched plots
(80.8% coverage). TF-IDF produced 31,609 word features, with no empty
feature vectors among these movies.

Candidates were scored by mean cosine similarity to the user's
training movies rated at least 4.0 that had plot features.
Previously rated movies were excluded. Missing-plot candidates
received zero text scores. Popularity broke ties and provided
the fallback when no liked movies had plot features.

Mean validation Recall@10:
- Overall: 3.761% across 165 users.
- Without training history: 4.185% across 140 users.
- With training history: 1.387% across 25 users.

Plot TF-IDF outperformed the genre-only model but underperformed
popularity and item-based collaborative filtering on this split.

Shared character names dominated some inspected similarities.
Their effect on aggregate recommendation performance remains untested.

Limitations include incomplete plot coverage, possible matching
errors, and only 25 evaluated users with training history.
The CMU text snapshot postdates the training cutoff, so this is
a retrospective experiment rather than a strictly historical simulation.

The test set remains unevaluated.

Let’s test logarithmic term frequency: repeated mentions will still matter, but each additional mention will contribute less.

In [47]:
tfidf_log_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    min_df=2,
    norm="l2",
    sublinear_tf=True
)

plot_tfidf_log = tfidf_log_vectorizer.fit_transform(
    text_movies["plot"]
)

print("Log-TF matrix shape:", plot_tfidf_log.shape)

Log-TF matrix shape: (5036, 31609)


In [48]:
log_text_rows = []

for user_id, user_positives in validation_positives.groupby("userId"):
    positive_movie_ids = user_positives["movieId"]

    recommendations = recommend_text_based(
        user_id=user_id,
        training_data=train_ratings,
        text_movies=text_movies,
        plot_tfidf=plot_tfidf_log,
        candidate_catalog=training_plot_catalog,
        n=10
    )

    hits = recommendations["movieId"].isin(
        positive_movie_ids
    ).sum()

    log_text_rows.append({
        "userId": user_id,
        "positive_movie_count": len(positive_movie_ids),
        "hits_at_10": hits,
        "recall_at_10": hits / len(positive_movie_ids)
    })

log_text_results = pd.DataFrame(log_text_rows)

log_text_results["has_training_history"] = (
    log_text_results["userId"].isin(train_ratings["userId"])
)

In [49]:
print("Users evaluated:", len(log_text_results))

print(
    "Original TF-IDF mean Recall@10:",
    text_based_results["recall_at_10"].mean()
)

print(
    "Log-TF mean Recall@10:",
    log_text_results["recall_at_10"].mean()
)

log_text_group_summary = (
    log_text_results.groupby("has_training_history")
    .agg(
        users=("userId", "size"),
        mean_recall_at_10=("recall_at_10", "mean")
    )
)

log_text_group_summary

Users evaluated: 165
Original TF-IDF mean Recall@10: 0.037607967210118935
Log-TF mean Recall@10: 0.036956304569506014


,users,mean_recall_at_10
has_training_history,,
False,140,0.041847
True,25,0.009569


tronger text baseline; collaborative filtering remains the strongest model overall.
Text model	Overall Recall@10	With training history
Original TF-IDF	3.761%	1.387%
Logarithmic TF-IDF	3.696%	0.957%


For users with history, recall decreased by 0.430 percentage points.

Our hypothesis was reasonable, but the experiment didn’t support adopting this change. Logarithmic weighting affects every repeated word, including useful overlaps between related films. This result doesn’t establish that character-name overlap is harmless.

Before closing the experiment, let’s check whether the decline affected many users or was concentrated in a few.

Compare the same users across both versions

In [50]:
paired_text_results = text_based_results.loc[
    text_based_results["has_training_history"],
    ["userId", "hits_at_10", "recall_at_10"]
].merge(
    log_text_results[
        ["userId", "hits_at_10", "recall_at_10"]
    ],
    on="userId",
    how="inner",
    suffixes=("_original", "_log"),
    validate="one_to_one"
)

paired_text_results["recall_change"] = (
    paired_text_results["recall_at_10_log"]
    - paired_text_results["recall_at_10_original"]
)

print("Users compared:", len(paired_text_results))

print(
    "Improved:",
    (
        paired_text_results["hits_at_10_log"]
        > paired_text_results["hits_at_10_original"]
    ).sum()
)

print(
    "Worsened:",
    (
        paired_text_results["hits_at_10_log"]
        < paired_text_results["hits_at_10_original"]
    ).sum()
)

print(
    "Unchanged:",
    (
        paired_text_results["hits_at_10_log"]
        == paired_text_results["hits_at_10_original"]
    ).sum()
)

Users compared: 25
Improved: 0
Worsened: 4
Unchanged: 21


In [51]:
changed_users = paired_text_results.loc[
    paired_text_results["recall_change"].ne(0)
]

display(
    changed_users.sort_values("recall_change")
)

,userId,hits_at_10_original,recall_at_10_original,hits_at_10_log,recall_at_10_log,recall_change
10,292,6,0.057143,0,0.000000,-0.057143
21,543,4,0.078431,2,0.039216,-0.039216
1,68,1,0.008403,0,0.000000,-0.008403
11,318,1,0.002762,0,0.000000,-0.002762


## Experiment: logarithmic term frequency

Hypothesis: Reducing the influence of repeated words might improve
recommendations, given the character-name dominance observed in
some movie similarities.

Change: Set sublinear_tf=True, replacing positive term counts with
1 + ln(count). All other vectorizer settings, data, recommendation
rules, and evaluation procedures remained unchanged.

Results:
- Overall mean Recall@10 decreased from 3.761% to 3.696%.
- Among 25 users with training history, mean Recall@10 decreased
  from 1.387% to 0.957%.
- Zero users improved, four worsened, and 21 had unchanged recall.
- The change lost 10 hits across those four users.
- Users without training history retained the popularity fallback.

Decision: Retain original TF-IDF as the stronger text baseline
on this validation split.

Interpretation: Dampening repetition did not improve the chosen
metric. This does not establish that character-name overlap is
harmless; logarithmic weighting changes all repeated terms.

The test set remains unevaluated.

In [52]:
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

training_plot_catalog.to_csv(
    processed_dir / "training_plot_catalog.csv",
    index=False
)